# AI AINAK — Notebook 4: Multimodal Transformer Fusion

**Branch 4:** Sliding-Window Timestamp Alignment, Modality Linear Projections, Multi-Head Self-Attention, Missing-Modality Masking, and Calibrated Hazard Priority & Alert Heads.

This notebook executes the trained Multimodal Transformer fusion model:
1. Loads the upstream feature outputs:
   - `vision_outputs.npz` (128-D features)
   - `audio_outputs.npz` (2,048-D features)
   - `gesture_outputs.npz` (128-D features)
2. Loads the trained **MultimodalTransformer** checkpoint (`fusion.pt`, 2.3 MB).
3. Synchronizes observation timestamps with sliding-window alignment ($\Delta t \le 1.5\,	ext{s}$).
4. Executes the trained Transformer encoder with multi-head self-attention.
5. Displays:
   - **128-D cross-modal context representation**.
   - **Priority score** (calibrated scalar $0.0$ to $1.0$, scaled to $0$–$100$).
   - **Confidence score** (calibrated scalar $0.0$ to $1.0$).
   - **Four alert logits** (`none`, `informational`, `caution`, `urgent`).
   - **Three modality contributions** (Vision, Audio, Gesture).
6. Evaluates missing-modality ablation robustness.
7. Saves outputs to `fusion_outputs.npz`.


In [ ]:
# Step 1: Install Dependencies
!pip install -q torch numpy matplotlib


In [ ]:
# Environment Detection & Path Resolution
import os, sys, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    from google.colab import drive
    # Mount Google Drive if available
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")
    
    # Locate project root
    drive_candidates = [
        Path("/content/drive/MyDrive/AINAK_Colab"),
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), Path.cwd())
else:
    print("Running in local environment.")
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root resolved to: {PROJECT_ROOT}")

# Helper to find checkpoint across standard folders
def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Checkpoint not found: {filename}. Checked: {[str(c) for c in candidates]}")

# Helper to find dataset / recording session
def find_session_media(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Media file {filename} for session {session_id} not found.")

# Ensure output directories exist
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)



## Checkpoint & Upstream Feature Ingestion
Load `fusion.pt` and the exported `.npz` files from Notebooks 1, 2, and 3.


In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
import time

fusion_path = find_checkpoint("fusion.pt", "fusion")
vision_npz = OUTPUTS_DIR / "vision_outputs.npz"
audio_npz = OUTPUTS_DIR / "audio_outputs.npz"
gesture_npz = OUTPUTS_DIR / "gesture_outputs.npz"

print(f"Loaded Transformer Checkpoint: {fusion_path} ({fusion_path.stat().st_size / 1e6:.2f} MB)")

# Verify upstream files exist (or load from recording fallback if running standalone)
if not (vision_npz.is_file() and audio_npz.is_file() and gesture_npz.is_file()):
    print("Notice: One or more .npz outputs not found in outputs/ directory.")
    print("Please execute Notebooks 01, 02, and 03 first to generate vision, audio, and gesture outputs.")
    # Fallback to session features if present
    fallback_features = PROJECT_ROOT / "datasets" / "recordings" / "session-multimodal-01" / "features.phase3.jsonl"
    if fallback_features.is_file():
        print(f"Loading pre-exported features from: {fallback_features}")
else:
    print(f"Found vision outputs: {vision_npz}")
    print(f"Found audio outputs: {audio_npz}")
    print(f"Found gesture outputs: {gesture_npz}")


## Multimodal Transformer Architecture Definition
Exact PyTorch implementation matching `ainak.models.fusion.transformer.MultimodalTransformer`.


In [ ]:
MODALITIES = ("vision", "audio", "gesture")
DEFAULT_FEATURE_DIMS = {"vision": 128, "audio": 2048, "gesture": 128}
ALERT_TYPES = ("none", "informational", "caution", "urgent")

class MultimodalTransformer(nn.Module):
    def __init__(self, feature_dims=None, dim=128, heads=4, layers=2, ff_dim=256, dropout=0.1, max_steps=128, modality_dropout=0.1):
        super().__init__()
        dims = feature_dims or DEFAULT_FEATURE_DIMS
        self.projections = nn.ModuleDict({m: nn.Sequential(nn.Linear(dims[m], dim), nn.LayerNorm(dim)) for m in MODALITIES})
        self.modality_embedding = nn.Parameter(torch.randn(3, dim) * 0.02)
        self.time_projection = nn.Sequential(nn.Linear(1, dim), nn.Tanh())
        self.position = nn.Parameter(torch.randn(max_steps, dim) * 0.02)
        
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=ff_dim,
                                           dropout=dropout, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, norm=nn.LayerNorm(dim), enable_nested_tensor=False)
        
        self.priority_head = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim//2, 1), nn.Sigmoid())
        self.confidence_head = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim//2, 1), nn.Sigmoid())
        self.alert_head = nn.Linear(dim, len(ALERT_TYPES))
        self.dim = dim
        self.max_steps = max_steps
        self.modality_dropout = modality_dropout

    def forward(self, features, mask, relative_times=None):
        b, t, _ = mask.shape
        tokens = []
        for idx, modality in enumerate(MODALITIES):
            raw = features[modality]
            token = self.projections[modality](raw)
            tokens.append(token + self.modality_embedding[idx])
        
        x = torch.stack(tokens, dim=2) # [B, T, 3, D]
        if relative_times is None:
            relative_times = torch.zeros(b, t, device=mask.device)
        time_signal = self.time_projection(relative_times.unsqueeze(-1)).unsqueeze(2)
        x = x + time_signal + self.position[:t].view(1, t, 1, self.dim)
        
        valid = mask.bool()
        flat = x.reshape(b, t * 3, self.dim)
        key_padding = ~valid.reshape(b, t * 3)
        empty = ~valid.reshape(b, -1).any(dim=1)
        
        if empty.any():
            flat = flat.clone()
            key_padding = key_padding.clone()
            flat[empty, 0] = 0.0
            key_padding[empty, 0] = False
            
        encoded = self.encoder(flat, src_key_padding_mask=key_padding)
        encoded = encoded.reshape(b, t, 3, self.dim)
        
        weights = valid.to(encoded.dtype)
        denom = weights.sum(dim=(1, 2), keepdim=False).clamp_min(1).unsqueeze(-1)
        context = (encoded * weights.unsqueeze(-1)).sum(dim=(1, 2)) / denom
        context = context.masked_fill(empty.unsqueeze(-1), 0.0)
        
        contribution = weights.sum(dim=1) / weights.sum(dim=(1, 2)).clamp_min(1).unsqueeze(-1)
        priority = self.priority_head(context).squeeze(-1).masked_fill(empty, 0.0)
        confidence = self.confidence_head(context).squeeze(-1).masked_fill(empty, 0.0)
        alert_logits = self.alert_head(context).masked_fill(empty.unsqueeze(-1), 0.0)
        
        return {
            "context": context,
            "priority": priority,
            "confidence": confidence,
            "alert_logits": alert_logits,
            "modality_contributions": contribution
        }

# Load weights
fusion_payload = torch.load(str(fusion_path), map_location="cpu", weights_only=False)
model_config = fusion_payload.get("model_config", {})
fusion_model = MultimodalTransformer(**model_config)
fusion_model.load_state_dict(fusion_payload["state_dict"], strict=True)
fusion_model.eval()

print("MultimodalTransformer loaded successfully.")
print(f"Encoder: {fusion_model.encoder.num_layers} Layers, 4 Heads | Embedding Dim: {fusion_model.dim}")


## Temporal Alignment & Forward Pass Execution
Pairs observations from the three branches with a sliding-window temporal aligner.


In [ ]:
# Load the 3 branch output matrices
v_data = np.load(str(vision_npz)) if vision_npz.is_file() else None
a_data = np.load(str(audio_npz)) if audio_npz.is_file() else None
g_data = np.load(str(gesture_npz)) if gesture_npz.is_file() else None

# Establish unified timeline anchors
if v_data is not None:
    timeline = v_data["timestamps"]
else:
    timeline = np.linspace(0, 3.0, 30)

fusion_timestamps = []
contexts = []
priorities = []
confidences = []
alert_logits_list = []
contributions = []
latencies_ms = []

print(f"Executing Transformer forward passes across {len(timeline)} timeline steps...")

for step_idx, t in enumerate(timeline):
    t_start = time.perf_counter()
    
    # Extract latest observations within window (delta_t <= 1.5s)
    # Vision
    has_v = False
    v_feat = np.zeros(128, dtype=np.float32)
    if v_data is not None:
        idx_v = np.searchsorted(v_data["timestamps"], t)
        if idx_v < len(v_data["timestamps"]) and abs(v_data["timestamps"][idx_v] - t) <= 1.5:
            v_feat = v_data["features"][idx_v]
            has_v = True

    # Audio
    has_a = False
    a_feat = np.zeros(2048, dtype=np.float32)
    if a_data is not None:
        idx_a = np.searchsorted(a_data["timestamps"], t)
        if idx_a < len(a_data["timestamps"]) and abs(a_data["timestamps"][idx_a] - t) <= 2.0:
            a_feat = a_data["features"][idx_a]
            has_a = True

    # Gesture
    has_g = False
    g_feat = np.zeros(128, dtype=np.float32)
    if g_data is not None:
        idx_g = np.searchsorted(g_data["timestamps"], t)
        if idx_g < len(g_data["timestamps"]) and abs(g_data["timestamps"][idx_g] - t) <= 1.5:
            g_feat = g_data["features"][idx_g]
            has_g = True

    # Construct input tensors: [B=1, T=1, D]
    feature_dict = {
        "vision": torch.from_numpy(v_feat[None, None, :]),
        "audio": torch.from_numpy(a_feat[None, None, :]),
        "gesture": torch.from_numpy(g_feat[None, None, :])
    }
    mask_tensor = torch.tensor([[[has_v, has_a, has_g]]], dtype=torch.bool)
    time_tensor = torch.tensor([[0.0]], dtype=torch.float32)

    with torch.no_grad():
        out = fusion_model(feature_dict, mask_tensor, time_tensor)

    t_latency = (time.perf_counter() - t_start) * 1000

    fusion_timestamps.append(t)
    contexts.append(out["context"][0].numpy())
    priorities.append(float(out["priority"][0].item()))
    confidences.append(float(out["confidence"][0].item()))
    alert_logits_list.append(out["alert_logits"][0].numpy())
    contributions.append(out["modality_contributions"][0].numpy())
    latencies_ms.append(t_latency)

print(f"Completed Transformer forward passes in {np.sum(latencies_ms):.1f} ms.")
print(f"Mean Forward Pass Latency: {np.mean(latencies_ms):.2f} ms | p95: {np.percentile(latencies_ms, 95):.2f} ms")


## Modality Contribution & Priority Trajectory Visualization


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))

# Priority & Confidence Progression
ax1.plot(fusion_timestamps, [p * 100 for p in priorities], label="Hazard Priority (0-100)", color="#8b5cf6", lw=2)
ax1.plot(fusion_timestamps, [c * 100 for c in confidences], label="Confidence (%)", color="#10b981", lw=1.5, ls="--")
ax1.axhline(60, color="#f43f5e", ls=":", label="Urgent Threshold (60)")
ax1.set_title("Transformer Calibrated Hazard Priority & Confidence Trajectory", fontsize=11, fontweight="bold")
ax1.set_xlabel("Session Timestamp (seconds)")
ax1.set_ylabel("Score")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

# Modality Contributions
contrib_arr = np.array(contributions)
ax2.plot(fusion_timestamps, contrib_arr[:, 0], label="Vision Contribution", color="#06b6d4", lw=1.8)
ax2.plot(fusion_timestamps, contrib_arr[:, 1], label="Audio Contribution", color="#f59e0b", lw=1.8)
ax2.plot(fusion_timestamps, contrib_arr[:, 2], label="Gesture Contribution", color="#10b981", lw=1.8)
ax2.set_title("Attention Modality Contribution Weights Across Time", fontsize=11, fontweight="bold")
ax2.set_xlabel("Session Timestamp (seconds)")
ax2.set_ylabel("Relative Attention Weight")
ax2.legend(loc="upper right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Missing-Modality Ablation Robustness Test
Demonstrates the key-padding masking capability across all 7 modality combinations.


In [ ]:
subsets = [
    ("Vision Only {V}", True, False, False),
    ("Audio Only {A}", False, True, False),
    ("Gesture Only {G}", False, False, True),
    ("Vision + Audio {V,A}", True, True, False),
    ("Vision + Gesture {V,G}", True, False, True),
    ("Audio + Gesture {A,G}", False, True, True),
    ("All Three {V,A,G}", True, True, True)
]

print("=== Transformer Missing Modality Robustness Evaluation ===")
for name, v_on, a_on, g_on in subsets:
    feature_dict = {
        "vision": torch.randn(1, 1, 128),
        "audio": torch.randn(1, 1, 2048),
        "gesture": torch.randn(1, 1, 128)
    }
    mask = torch.tensor([[[v_on, a_on, g_on]]], dtype=torch.bool)
    with torch.no_grad():
        out = fusion_model(feature_dict, mask)
    
    p = float(out["priority"][0].item()) * 100
    c = float(out["confidence"][0].item()) * 100
    logits = out["alert_logits"][0].numpy()
    pred_alert = ALERT_TYPES[int(logits.argmax())]
    print(f"{name:<24} | Priority={p:5.1f}% | Conf={c:5.1f}% | Alert={pred_alert.upper()}")


## Export Fusion Outputs to `fusion_outputs.npz`


In [ ]:
fusion_out_path = OUTPUTS_DIR / "fusion_outputs.npz"

np.savez_compressed(
    str(fusion_out_path),
    timestamps=np.array(fusion_timestamps, dtype=np.float64),
    context=np.array(contexts, dtype=np.float32),
    priorities=np.array(priorities, dtype=np.float32),
    confidences=np.array(confidences, dtype=np.float32),
    alert_logits=np.array(alert_logits_list, dtype=np.float32),
    modality_contributions=np.array(contributions, dtype=np.float32),
    latencies_ms=np.array(latencies_ms, dtype=np.float32)
)

print(f"Successfully exported fusion outputs to: {fusion_out_path}")
print(f"File size: {fusion_out_path.stat().st_size / 1e3:.1f} KB")
print(f"Context shape: {np.array(contexts).shape} (Matches 128-D context representation contract)")
